# 04_limpieza_06_crear_diccionario_autores_unam

Esta libreta crea el catálogo preliminar de autores para la fase de normalización de `Autor_norm`.

Genera **únicamente dos archivos**:

1. `diccionario_alias_autores_unam.csv`
2. `autores_unicos_sin_coincidencia.csv`

No modifica todavía la base principal, no deduplica y no crea un archivo de autores normalizados. Los autores sin coincidencia segura quedan preparados para búsqueda manual con título, afiliaciones, DOI, URL, ISBN e ISSN.

In [1]:
from __future__ import annotations

from pathlib import Path
from collections import Counter, defaultdict
from difflib import SequenceMatcher
import re
import unicodedata
import pandas as pd

# ---------------------------------------------------------------------
# 1. Configuración de rutas
# ---------------------------------------------------------------------
RUTA_BASE = Path(r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv")
RUTA_TUTORA = Path(r"C:\Users\hazar\Desktop\PROYECTO\00_control\UNAM_Completo_Corregido.xlsx")
CARPETA_SALIDA = Path(r"C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres")

if not RUTA_BASE.exists() and Path("/mnt/data/integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv").exists():
    RUTA_BASE = Path("/mnt/data/integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv")
if not RUTA_TUTORA.exists() and Path("/mnt/data/UNAM_Completo_Corregido.xlsx").exists():
    RUTA_TUTORA = Path("/mnt/data/UNAM_Completo_Corregido.xlsx")
if not str(CARPETA_SALIDA).startswith("C:"):
    pass
if str(RUTA_BASE).startswith("/mnt/data"):
    CARPETA_SALIDA = Path("/mnt/data/03_Nomralizar_Nombres")

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

SALIDA_DICCIONARIO = CARPETA_SALIDA / "diccionario_alias_autores_unam.csv"
SALIDA_AUTORES_SIN_MATCH = CARPETA_SALIDA / "autores_unicos_sin_coincidencia.csv"

COLUMNAS_CANONICAS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

PARTICULAS = {
    "de", "del", "la", "las", "los", "y", "e", "da", "do", "dos", "das",
    "van", "von", "du", "di", "of", "the", "al", "ibn",
}

# ---------------------------------------------------------------------
# 2. Funciones de limpieza y comparación de nombres
# ---------------------------------------------------------------------
def limpiar_espacios(valor: object) -> str:
    texto = "" if valor is None else str(valor)
    texto = texto.replace("\ufeff", " ").replace("\n", " ").replace("\r", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", texto).strip()


def quitar_acentos(texto: str) -> str:
    texto = "" if texto is None else str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def clave_norm(texto: object) -> str:
    texto = quitar_acentos(limpiar_espacios(texto)).lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def remover_ruido_autor(texto: object) -> str:
    texto = limpiar_espacios(texto)
    texto = re.sub(r"https?://\S+|\S+@\S+", " ", texto)
    texto = re.sub(r"\bORCID\b.*$", " ", texto, flags=re.I)
    texto = re.sub(r"\[[^\]]*\]|\([^)]*\)", " ", texto)
    texto = re.sub(r"[*†‡§]+", " ", texto)
    texto = re.sub(r"\d+", " ", texto)
    return limpiar_espacios(texto)


def nombre_sin_puntuacion(texto: object) -> str:
    texto = remover_ruido_autor(texto)
    texto = texto.replace("‐", "-").replace("‑", "-").replace("–", "-").replace("—", "-")
    texto = texto.replace("-", " ")
    texto = re.sub(r"[.,;:!¡?¿\"“”‘’`´]", " ", texto)
    return limpiar_espacios(texto)


def autor_a_display(valor: object) -> str:
    """
    Convierte una forma cruda a una forma limpia para comparación.
    Ejemplos:
    - Pérez, Juan C. -> Juan C Pérez
    - J. Pérez -> J Pérez
    - José-Alberto Macías-Vargas -> José Alberto Macías Vargas
    """
    raw = remover_ruido_autor(valor)
    if "," in raw:
        apellido, nombres = raw.split(",", 1)
        return nombre_sin_puntuacion(f"{nombres.strip()} {apellido.strip()}")
    return nombre_sin_puntuacion(raw)


def tokens(texto: object) -> list[str]:
    return [t for t in clave_norm(texto).split() if t]


def es_inicial(token: str) -> bool:
    return len(token) == 1 and token.isalpha()


def dividir_autor_crudo(raw: object) -> tuple[str, list[str], list[str]]:
    """Devuelve display limpio, tokens aproximados de nombres y tokens de apellidos."""
    raw_clean = remover_ruido_autor(raw)
    if "," in raw_clean:
        apellido, nombres = raw_clean.split(",", 1)
        return autor_a_display(raw_clean), tokens(nombres), tokens(apellido)

    display = autor_a_display(raw_clean)
    toks = tokens(display)
    if not toks:
        return display, [], []

    # Si empieza con iniciales, esas se toman como nombres y el resto como apellidos.
    i = 0
    while i < len(toks) and es_inicial(toks[i]):
        i += 1
    if i > 0:
        return display, toks[:i], toks[i:]

    # Sin coma, sólo para scoring aproximado.
    if len(toks) >= 4:
        return display, toks[:-2], toks[-2:]
    if len(toks) >= 2:
        return display, toks[:-1], toks[-1:]
    return display, toks, []


def apellido_keys_desde_display(display: str) -> set[str]:
    toks = [t for t in tokens(display) if t not in PARTICULAS]
    keys = set()
    if toks:
        keys.add(toks[-1])
    if len(toks) >= 2:
        keys.add(" ".join(toks[-2:]))
    if len(toks) >= 3:
        keys.add(" ".join(toks[-3:]))
    return keys


def apellido_keys_desde_raw(raw: object) -> set[str]:
    display, _, apellidos = dividir_autor_crudo(raw)
    apellidos_core = [t for t in apellidos if t not in PARTICULAS]
    keys = set()
    if apellidos_core:
        keys.add(apellidos_core[-1])
        if len(apellidos_core) >= 2:
            keys.add(" ".join(apellidos_core[-2:]))
        if len(apellidos_core) >= 3:
            keys.add(" ".join(apellidos_core[-3:]))
    else:
        keys |= apellido_keys_desde_display(display)
    return keys


def iniciales_texto(toks: list[str]) -> str:
    return "".join(t[0] for t in toks if t and t not in PARTICULAS)


def parece_con_iniciales(raw: object) -> bool:
    display, nombres, _ = dividir_autor_crudo(raw)
    toks = tokens(display)
    if not toks:
        return True
    if any(es_inicial(t) for t in nombres):
        return True
    if sum(es_inicial(t) for t in toks) >= 1 and len(toks) <= 4:
        return True
    return False


def primer_nombre_completo(raw: object) -> str:
    _, nombres, _ = dividir_autor_crudo(raw)
    for t in nombres:
        if t not in PARTICULAS and not es_inicial(t):
            return t
    for t in tokens(autor_a_display(raw)):
        if t not in PARTICULAS and not es_inicial(t):
            return t
    return ""


def elegir_mejor_nombre(nombres: list[str], preferidos: set[str] | None = None) -> str:
    preferidos = preferidos or set()
    nombres = [autor_a_display(n) for n in nombres if limpiar_espacios(n)]
    if not nombres:
        return ""
    conteo = Counter(nombres)

    def score(n: str):
        toks = tokens(n)
        tiene_acento = quitar_acentos(n) != n
        inicial = parece_con_iniciales(n)
        return (
            5 if n in preferidos else 0,
            0 if inicial else 3,
            1 if tiene_acento else 0,
            len(toks),
            conteo[n],
            len(n),
        )

    return sorted(set(nombres), key=score, reverse=True)[0]


def doi_limpio(valor: object) -> str:
    texto = limpiar_espacios(valor).lower()
    texto = re.sub(r"^https?://(dx\.)?doi\.org/", "", texto)
    texto = re.sub(r"^doi\s*[:/]\s*", "", texto)
    m = re.search(r"10\.\S+", texto)
    return m.group(0).strip(".,;)") if m else ""


def clave_afiliacion(valor: object) -> str:
    return clave_norm(valor)


def nombres_compatibles(base_raw: object, tutor_name: str) -> tuple[float, str]:
    """
    Puntaje de compatibilidad entre un autor de la base actual y un autor de la tutora.
    No modifica nombres finales; sólo decide si hay evidencia suficiente.
    """
    base_display, base_nombres, base_apellidos = dividir_autor_crudo(base_raw)
    tutor_display, tutor_nombres, tutor_apellidos = dividir_autor_crudo(tutor_name)
    kb = clave_norm(base_display)
    kt = clave_norm(tutor_display)

    if not kb or not kt:
        return 0.0, "vacio"
    if kb == kt:
        return 1.0, "exacto_normalizado"

    tutor_tokens = set(tokens(tutor_display))
    base_surn_core = [t for t in base_apellidos if t not in PARTICULAS]

    # Si el autor base venía como Apellido, Nombre, los apellidos deben existir en el candidato.
    if base_surn_core and not all(t in tutor_tokens for t in base_surn_core):
        return 0.0, "apellido_no_coincide"

    # Validación de nombres o iniciales.
    if base_nombres:
        ok_nombre = True
        for g in base_nombres:
            if g in PARTICULAS:
                continue
            if es_inicial(g):
                if not any(t.startswith(g) for t in tutor_tokens if t not in PARTICULAS):
                    ok_nombre = False
                    break
            else:
                if g not in tutor_tokens:
                    ok_nombre = False
                    break
        if not ok_nombre:
            return 0.0, "nombre_no_coincide"

    sb = set(tokens(base_display))
    st = set(tokens(tutor_display))
    jacc = len(sb & st) / max(1, len(sb | st))
    seq = SequenceMatcher(None, kb, kt).ratio()
    score = max(jacc * 0.95, seq * 0.65)

    if base_surn_core and base_nombres:
        score = max(score, 0.90)
    if sb and sb.issubset(st):
        score = max(score, 0.93)
    if st and st.issubset(sb):
        score = max(score, 0.91)

    return min(score, 1.0), f"seq={seq:.2f};jacc={jacc:.2f}"

# ---------------------------------------------------------------------
# 3. Carga y validación de archivos
# ---------------------------------------------------------------------
def cargar_archivos() -> tuple[pd.DataFrame, pd.DataFrame]:
    base = pd.read_csv(RUTA_BASE, dtype=str, keep_default_na=False, encoding="utf-8-sig")
    tutora = pd.read_excel(RUTA_TUTORA, dtype=str, keep_default_na=False)

    if list(base.columns) != COLUMNAS_CANONICAS:
        raise ValueError(f"La base actual no conserva las columnas canónicas exactas: {list(base.columns)}")
    if "Autor_norm" not in tutora.columns:
        raise ValueError("La base de la tutora no tiene la columna Autor_norm.")

    for col in COLUMNAS_CANONICAS:
        base[col] = base[col].map(limpiar_espacios)
    tutora["Autor_norm"] = tutora["Autor_norm"].map(limpiar_espacios)

    return base, tutora

# ---------------------------------------------------------------------
# 4. Construir catálogo de autores
# ---------------------------------------------------------------------
def construir_catalogo_tutora(tutora: pd.DataFrame):
    tutora = tutora.copy()
    tutora["autor_canon_tutora"] = tutora["Autor_norm"].map(autor_a_display)
    tutora = tutora[tutora["autor_canon_tutora"].str.strip().ne("")].copy()

    # Si hay variantes exactas normalizadas en la base de la tutora, escoger la forma más completa/preferible.
    tutor_preferido = {}
    for key, vals in tutora.groupby(tutora["autor_canon_tutora"].map(clave_norm))["autor_canon_tutora"]:
        tutor_preferido[key] = elegir_mejor_nombre(vals.tolist(), preferidos=set(vals.tolist()))

    autores_tutora = sorted(set(tutor_preferido.values()), key=clave_norm)
    tutor_exact = {clave_norm(a): a for a in autores_tutora}

    tutor_afiliaciones = defaultdict(set)
    tutor_dois = defaultdict(set)
    tutor_por_doi = defaultdict(list)

    for _, r in tutora.iterrows():
        canon = tutor_preferido.get(clave_norm(r["autor_canon_tutora"]), r["autor_canon_tutora"])
        for col in ["Afiliacion1", "Afiliacion2"]:
            if col in tutora.columns and limpiar_espacios(r.get(col, "")):
                tutor_afiliaciones[canon].add(clave_afiliacion(r.get(col, "")))
        d = doi_limpio(r.get("Doi", "")) if "Doi" in tutora.columns else ""
        if d:
            tutor_dois[canon].add(d)
            tutor_por_doi[d].append(canon)

    tutor_por_apellido = defaultdict(set)
    for canon in autores_tutora:
        for k in apellido_keys_desde_display(canon):
            tutor_por_apellido[k].add(canon)

    return tutora, autores_tutora, tutor_exact, tutor_afiliaciones, tutor_dois, tutor_por_doi, tutor_por_apellido

# ---------------------------------------------------------------------
# 5. Resumen de autores únicos de la base actual
# ---------------------------------------------------------------------
def unir_valores_unicos(valores: list[str], limite: int = 10, separador: str = " | ") -> str:
    salida = []
    vistos = set()
    for v in valores:
        v = limpiar_espacios(v)
        if not v:
            continue
        k = clave_norm(v)
        if k not in vistos:
            vistos.add(k)
            salida.append(v)
        if len(salida) >= limite:
            break
    return separador.join(salida)


def construir_autores_base(base: pd.DataFrame) -> list[dict]:
    registros = []
    for autor, g in base.groupby("Autor_norm", dropna=False):
        afiliaciones = set()
        for col in ["Afiliacion1", "Afiliacion2"]:
            afiliaciones |= {clave_afiliacion(x) for x in g[col].tolist() if limpiar_espacios(x)}
        dois = {doi_limpio(x) for x in g["Doi"].tolist() if doi_limpio(x)}
        registros.append({
            "alias": autor,
            "autor_propuesto_limpio": autor_a_display(autor),
            "key": clave_norm(autor_a_display(autor)),
            "n_filas_base": int(g.shape[0]),
            "n_articulos_base": int(g["indice"].nunique()),
            "indices": unir_valores_unicos(g["indice"].tolist(), limite=20),
            "titulos_asociados": unir_valores_unicos(g["Titulo"].tolist(), limite=8),
            "afiliacion1_asociada": unir_valores_unicos(g["Afiliacion1"].tolist(), limite=8),
            "afiliacion2_asociada": unir_valores_unicos(g["Afiliacion2"].tolist(), limite=8),
            "isbn_asociado": unir_valores_unicos(g["ISBN"].tolist(), limite=8),
            "issn_asociado": unir_valores_unicos(g["ISSN"].tolist(), limite=8),
            "doi_asociado": unir_valores_unicos(g["Doi"].tolist(), limite=8),
            "url_asociada": unir_valores_unicos(g["URL"].tolist(), limite=8),
            "area_asociada": unir_valores_unicos(g["Area"].tolist(), limite=8),
            "dois": dois,
            "afiliaciones": afiliaciones,
            "afiliaciones_txt": " | ".join(sorted(afiliaciones)[:12]),
        })
    return registros

# ---------------------------------------------------------------------
# 6. Matching base actual
# ---------------------------------------------------------------------
def resolver_mapeos(
    autores_base: list[dict],
    tutor_exact: dict,
    tutor_afiliaciones: dict,
    tutor_por_doi: dict,
    tutor_por_apellido: dict,
) -> pd.DataFrame:
    filas = []

    for rec in autores_base:
        alias = rec["alias"]
        propuesto = rec["autor_propuesto_limpio"]
        key = rec["key"]

        # 1. Coincidencia exacta normalizada.
        if key in tutor_exact:
            canon = tutor_exact[key]
            filas.append({
                **rec,
                "autor_normalizado": canon,
                "estatus": "match_tutora",
                "origen": "tutora_exacta",
                "confianza": "alta",
                "score": 1.0,
                "motivo": "exacto_normalizado",
                "candidatos_tutora": canon,
                "n_candidatos": 1,
            })
            continue

        # 2. DOI compartido y nombre compatible. En esta base puede ser raro, pero es evidencia fuerte.
        doi_candidates = Counter()
        for d in rec["dois"]:
            for c in tutor_por_doi.get(d, []):
                sc, _ = nombres_compatibles(alias, c)
                if sc >= 0.80:
                    doi_candidates[c] += 1
        if doi_candidates:
            canon, _ = doi_candidates.most_common(1)[0]
            filas.append({
                **rec,
                "autor_normalizado": canon,
                "estatus": "match_tutora",
                "origen": "tutora_doi_compartido",
                "confianza": "alta",
                "score": 0.98,
                "motivo": "mismo_doi_y_nombre_compatible",
                "candidatos_tutora": canon,
                "n_candidatos": len(doi_candidates),
            })
            continue

        # 3. Búsqueda por apellido + validación de nombres/iniciales + afiliación.
        cand_set = set()
        for sk in apellido_keys_desde_raw(alias):
            cand_set |= tutor_por_apellido.get(sk, set())

        scored = []
        for cand in cand_set:
            sc, why = nombres_compatibles(alias, cand)
            if sc <= 0:
                continue
            aff_overlap = bool(rec["afiliaciones"] & tutor_afiliaciones.get(cand, set()))
            if aff_overlap:
                sc = min(1.0, sc + 0.06)
                why += ";afiliacion_coincide"
            scored.append((cand, sc, why, aff_overlap))
        scored.sort(key=lambda x: x[1], reverse=True)

        if scored:
            top = scored[0]
            second = scored[1] if len(scored) > 1 else ("", 0.0, "", False)
            low_info = parece_con_iniciales(alias)
            margin = top[1] - second[1]

            aceptar = False
            confianza = "media"
            if top[1] >= 0.98 and margin >= 0.08 and not low_info:
                aceptar = True
                confianza = "alta"
            elif top[1] >= 0.94 and margin >= 0.08 and not low_info:
                aceptar = True
                confianza = "media"
            elif top[1] >= 0.92 and top[3] and margin >= 0.03:
                # Si hay solapamiento de afiliación, se permite incluso con iniciales.
                aceptar = True
                confianza = "media"

            if aceptar:
                filas.append({
                    **rec,
                    "autor_normalizado": top[0],
                    "estatus": "match_tutora",
                    "origen": "tutora_probable",
                    "confianza": confianza,
                    "score": round(top[1], 3),
                    "motivo": top[2],
                    "candidatos_tutora": " | ".join([f"{c}:{s:.3f}" for c, s, _, _ in scored[:5]]),
                    "n_candidatos": len(scored),
                })
            else:
                filas.append({
                    **rec,
                    "autor_normalizado": propuesto,
                    "estatus": "sin_coincidencia_segura",
                    "origen": "revision_manual",
                    "confianza": "revision",
                    "score": round(top[1], 3),
                    "motivo": f"revision_top={top[0]}:{top[2]}; segundo={second[0]}:{second[1]:.2f}; iniciales_o_baja_info={low_info}; margen={margin:.3f}",
                    "candidatos_tutora": " | ".join([f"{c}:{s:.3f}" for c, s, _, _ in scored[:5]]),
                    "n_candidatos": len(scored),
                })
            continue

        # 4. Sin candidato.
        filas.append({
            **rec,
            "autor_normalizado": propuesto,
            "estatus": "sin_coincidencia",
            "origen": "nuevo_o_no_en_tutora",
            "confianza": "nuevo",
            "score": 0.0,
            "motivo": "sin_match_tutora",
            "candidatos_tutora": "",
            "n_candidatos": 0,
        })

    mapeo = pd.DataFrame(filas)
    return mapeo

# ---------------------------------------------------------------------
# 7. Crear los dos archivos finales
# ---------------------------------------------------------------------
def construir_salidas(mapeo: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    mapeo = mapeo.copy()

    # Diccionario de alias seguros: sólo coincidencias.
    dic_rows = []
    seguros = mapeo[mapeo["estatus"].eq("match_tutora")].copy()

    for _, r in seguros.iterrows():
        dic_rows.append({
            "alias": r["alias"],
            "autor_normalizado": r["autor_normalizado"],
            "confianza": r["confianza"],
            "origen": r["origen"],
            "score": r["score"],
            "n_filas_base": r["n_filas_base"],
            "n_articulos_base": r["n_articulos_base"],
        })
        if clave_norm(r["alias"]) != clave_norm(r["autor_propuesto_limpio"]):
            dic_rows.append({
                "alias": r["autor_propuesto_limpio"],
                "autor_normalizado": r["autor_normalizado"],
                "confianza": r["confianza"],
                "origen": r["origen"] + "_display",
                "score": r["score"],
                "n_filas_base": r["n_filas_base"],
                "n_articulos_base": r["n_articulos_base"],
            })

    diccionario = pd.DataFrame(dic_rows)
    if diccionario.empty:
        diccionario = pd.DataFrame(columns=[
            "alias", "autor_normalizado", "confianza", "origen", "score", "n_filas_base", "n_articulos_base"
        ])
    else:
        diccionario = diccionario.drop_duplicates().copy()
        diccionario["alias_key"] = diccionario["alias"].map(clave_norm)

        # Si un alias normalizado apunta a más de un autor, se elimina del diccionario y se manda a pendientes.
        conflictos = diccionario.groupby("alias_key")["autor_normalizado"].nunique()
        conflict_keys = set(conflictos[conflictos > 1].index)
        diccionario_seguro = diccionario[~diccionario["alias_key"].isin(conflict_keys)].copy()
        diccionario_seguro = diccionario_seguro.drop(columns=["alias_key"]).sort_values(["autor_normalizado", "alias"]).reset_index(drop=True)
    
        if conflict_keys:
            # Convertir autores afectados por conflicto a pendientes.
            alias_conflictivos = set(diccionario[diccionario["alias_key"].isin(conflict_keys)]["alias"].map(clave_norm))
            mapeo.loc[mapeo["alias"].map(clave_norm).isin(alias_conflictivos), "estatus"] = "conflicto_alias"
            mapeo.loc[mapeo["alias"].map(clave_norm).isin(alias_conflictivos), "motivo"] = "alias_normalizado_apunta_a_mas_de_un_autor"
        diccionario = diccionario_seguro

    pendientes = mapeo[~mapeo["estatus"].eq("match_tutora")].copy()

    columnas_pendientes = [
        "alias",
        "autor_propuesto_limpio",
        "estatus",
        "confianza",
        "score",
        "motivo",
        "candidatos_tutora",
        "n_candidatos",
        "n_filas_base",
        "n_articulos_base",
        "indices",
        "titulos_asociados",
        "afiliacion1_asociada",
        "afiliacion2_asociada",
        "isbn_asociado",
        "issn_asociado",
        "doi_asociado",
        "url_asociada",
        "area_asociada",
    ]

    pendientes = pendientes[columnas_pendientes].rename(columns={
        "alias": "Autor_norm_original",
        "titulos_asociados": "Titulo_asociado",
        "afiliacion1_asociada": "Afiliacion1",
        "afiliacion2_asociada": "Afiliacion2",
        "isbn_asociado": "ISBN",
        "issn_asociado": "ISSN",
        "doi_asociado": "Doi",
        "url_asociada": "URL",
        "area_asociada": "Area",
    })

    pendientes = pendientes.sort_values(["estatus", "n_filas_base", "Autor_norm_original"], ascending=[True, False, True]).reset_index(drop=True)

    return diccionario, pendientes

# ---------------------------------------------------------------------
# 8. Ejecución principal
# ---------------------------------------------------------------------
def main() -> tuple[pd.DataFrame, pd.DataFrame]:
    base, tutora = cargar_archivos()
    (
        tutora_proc,
        autores_tutora,
        tutor_exact,
        tutor_afiliaciones,
        tutor_dois,
        tutor_por_doi,
        tutor_por_apellido,
    ) = construir_catalogo_tutora(tutora)

    autores_base = construir_autores_base(base)
    mapeo = resolver_mapeos(
        autores_base=autores_base,
        tutor_exact=tutor_exact,
        tutor_afiliaciones=tutor_afiliaciones,
        tutor_por_doi=tutor_por_doi,
        tutor_por_apellido=tutor_por_apellido,
    )

    diccionario, pendientes = construir_salidas(mapeo)

    diccionario.to_csv(SALIDA_DICCIONARIO, index=False, encoding="utf-8-sig")
    pendientes.to_csv(SALIDA_AUTORES_SIN_MATCH, index=False, encoding="utf-8-sig")

    # Validaciones básicas.
    if list(base.columns) != COLUMNAS_CANONICAS:
        raise AssertionError("La base de entrada perdió columnas canónicas.")
    if set(diccionario.columns) != {"alias", "autor_normalizado", "confianza", "origen", "score", "n_filas_base", "n_articulos_base"}:
        raise AssertionError("El diccionario no tiene las columnas esperadas.")
    if "Autor_norm_original" not in pendientes.columns:
        raise AssertionError("El archivo de pendientes no contiene Autor_norm_original.")

    print("Proceso terminado.")
    print(f"Base actual: {RUTA_BASE}")
    print(f"Base tutora: {RUTA_TUTORA}")
    print(f"Autores únicos en base actual: {len(autores_base):,}")
    print(f"Autores únicos en base tutora: {len(autores_tutora):,}")
    print(f"Aliases seguros generados: {len(diccionario):,}")
    print(f"Autores únicos sin coincidencia segura: {len(pendientes):,}")
    print("\nArchivos generados:")
    print(SALIDA_DICCIONARIO)
    print(SALIDA_AUTORES_SIN_MATCH)

    return diccionario, pendientes

if __name__ == "__main__":
    main()


Proceso terminado.
Base actual: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv
Base tutora: C:\Users\hazar\Desktop\PROYECTO\00_control\UNAM_Completo_Corregido.xlsx
Autores únicos en base actual: 1,112
Autores únicos en base tutora: 1,902
Aliases seguros generados: 525
Autores únicos sin coincidencia segura: 829

Archivos generados:
C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres\diccionario_alias_autores_unam.csv
C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres\autores_unicos_sin_coincidencia.csv
